# Notebook 1 – MER-TRANS Dataset Analysis (ES + IT)

**Task:** MER-TRANS 2026 – Multilingual Easy-to-Read Text Simplification  
**Objective:** Explore trial, test, and gold datasets for Spanish and Italian.

---


> The test set does not include `simplified_text` as it was released without references during the competition.

---

### Expected data directory structure

```
data/
├── trial/
│   ├── es_trial_document.csv
│   ├── it_trial_document.csv
│   └── cat_trial_document.csv
├── test/
│   ├── es_test_data.csv
│   └── it_test_data.csv
└── gold/
    ├── es_gold_data.csv
    └── it_gold_data.csv
```

## Installation

In [ ]:
!pip install pandas numpy matplotlib nltk

## 1. Imports and data loading

In [ ]:
import re
import pandas as pd
import numpy as np
import nltk
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import word_tokenize

# Paths: update to match your local directory structure
ES_TRIAL_PATH = "data/trial/es_trial_document.csv"
IT_TRIAL_PATH = "data/trial/it_trial_document.csv"
ES_TEST_PATH  = "data/test/es_test_data.csv"
IT_TEST_PATH  = "data/test/it_test_data.csv"
ES_GOLD_PATH  = "data/gold/es_gold_data.csv"
IT_GOLD_PATH  = "data/gold/it_gold_data.csv"

es_trial = pd.read_csv(ES_TRIAL_PATH)
it_trial = pd.read_csv(IT_TRIAL_PATH)
es_test  = pd.read_csv(ES_TEST_PATH)
it_test  = pd.read_csv(IT_TEST_PATH)
es_gold  = pd.read_csv(ES_GOLD_PATH)
it_gold  = pd.read_csv(IT_GOLD_PATH)

print(f"ES trial: {len(es_trial)} segments, {es_trial['document_id'].nunique()} document(s)")
print(f"IT trial: {len(it_trial)} segments, {it_trial['document_id'].nunique()} document(s)")
print(f"ES test:  {len(es_test)} segments,  {es_test['document_id'].nunique()} documents")
print(f"IT test:  {len(it_test)} segments,  {it_test['document_id'].nunique()} documents")
print(f"ES gold:  {len(es_gold)} segments,  {es_gold['document_id'].nunique()} documents")
print(f"IT gold:  {len(it_gold)} segments,  {it_gold['document_id'].nunique()} documents")

## 2. Helper functions

>  MER-TRANS segments use line breaks (`\n`) as sentence
> separators rather than full stops. The number of non-empty lines is counted
> instead of using a standard sentence tokeniser.

In [ ]:
def clean_text(text: str) -> str:
    text = str(text).replace('\xa0', ' ')
    text = re.sub(r' +', ' ', text)
    return text.strip()

def count_tokens(text: str, lang: str = 'spanish') -> int:
    return len(word_tokenize(text, language=lang))

def count_lines(text: str) -> int:
    return len([l for l in str(text).split('\n') if l.strip()])

def add_metrics(df: pd.DataFrame, lang: str, has_simplified: bool) -> pd.DataFrame:
    df = df.copy()
    df['original_clean'] = df['original_text'].astype(str).apply(clean_text)
    df['orig_tokens'] = df['original_clean'].apply(lambda x: count_tokens(x, lang))
    df['orig_lines']  = df['original_text'].apply(count_lines)
    df['orig_chars']  = df['original_clean'].apply(len)
    if has_simplified:
        df['simplified_clean'] = df['simplified_text'].astype(str).apply(clean_text)
        df['simp_tokens'] = df['simplified_clean'].apply(lambda x: count_tokens(x, lang))
        df['simp_lines']  = df['simplified_text'].apply(count_lines)
        df['simp_chars']  = df['simplified_clean'].apply(len)
        df['ratio_tokens'] = (df['simp_tokens'] / df['orig_tokens']).round(3)
    return df

es_trial = add_metrics(es_trial, 'spanish', has_simplified=True)
it_trial = add_metrics(it_trial, 'italian', has_simplified=True)
es_test  = add_metrics(es_test,  'spanish', has_simplified=False)
it_test  = add_metrics(it_test,  'italian', has_simplified=False)

print("Metrics computed for all splits.")

## 3. Statistics Trial (ES and IT)

In [ ]:
for name, df in [('ES Trial', es_trial), ('IT Trial', it_trial)]:
    print(f"\n{'='*60}")
    print(f"  {name}  ({len(df)} segments, {df['document_id'].nunique()} document(s))")
    print(f"{'='*60}")
    cols = ['orig_tokens','simp_tokens','orig_lines','simp_lines','orig_chars','simp_chars']
    stats = df[cols].describe(percentiles=[0.25,0.5,0.75]).round(2)
    stats.index = ['count','mean','std','min','Q1','median','Q3','max']
    print(stats.to_string())
    print(f"\nMean simplification ratio (simp/orig tokens): {df['ratio_tokens'].mean():.3f}")
    print(f"Ratio > 1 in {(df['ratio_tokens'] > 1).sum()}/{len(df)} segments")

## 4. Statistics – Test (ES and IT)

In [ ]:
for name, df in [('ES Test', es_test), ('IT Test', it_test)]:
    print(f"\n{'='*60}")
    print(f"  {name}  ({len(df)} segments, {df['document_id'].nunique()} documents)")
    print(f"{'='*60}")
    stats = df[['orig_tokens','orig_lines','orig_chars']].describe(
        percentiles=[0.25,0.5,0.75]).round(2)
    stats.index = ['count','mean','std','min','Q1','median','Q3','max']
    print(stats.to_string())

## 5. Test set length distribution (ES and IT)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("MER-TRANS – Test Dataset\nToken length distribution by language",
             fontsize=13, fontweight='bold')

for ax, df, title, color in [(axes[0], es_test, 'Spanish (ES)', '#4C72B0'),
                              (axes[1], it_test, 'Italian (IT)', '#55A868')]:
    ax.hist(df['orig_tokens'], bins=20, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df['orig_tokens'].mean(),   color='red',    linestyle='--', linewidth=1.5,
               label=f"Mean: {df['orig_tokens'].mean():.1f}")
    ax.axvline(df['orig_tokens'].median(), color='orange', linestyle=':',  linewidth=1.5,
               label=f"Median: {df['orig_tokens'].median():.1f}")
    ax.set_title(title)
    ax.set_xlabel('No. tokens')
    ax.set_ylabel('Frequency')
    ax.legend()

plt.tight_layout()
plt.show()

## 6. Cross-language and cross-split comparison

In [ ]:
labels = ['ES Trial', 'IT Trial', 'ES Test', 'IT Test']
means  = [es_trial['orig_tokens'].mean(), it_trial['orig_tokens'].mean(),
          es_test['orig_tokens'].mean(),  it_test['orig_tokens'].mean()]
colors = ['#4C72B0', '#55A868', '#4C72B0', '#55A868']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, means, color=colors, edgecolor='white', width=0.5)
ax.bar_label(bars, fmt='%.1f', padding=3)
ax.set_ylabel('Mean tokens (original text)')
ax.set_title('Mean original segment length\nby split and language')
ax.legend(handles=[Patch(color='#4C72B0', label='Spanish'),
                   Patch(color='#55A868', label='Italian')])
plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    'Split':             labels,
    'Segments':          [len(es_trial), len(it_trial), len(es_test), len(it_test)],
    'Documents':         [es_trial['document_id'].nunique(), it_trial['document_id'].nunique(),
                          es_test['document_id'].nunique(),  it_test['document_id'].nunique()],
    'Mean tokens':       [round(m, 1) for m in means],
    'Median tokens':     [es_trial['orig_tokens'].median(), it_trial['orig_tokens'].median(),
                          es_test['orig_tokens'].median(),  it_test['orig_tokens'].median()],
    'Mean lines (orig)': [round(es_trial['orig_lines'].mean(),1), round(it_trial['orig_lines'].mean(),1),
                          round(es_test['orig_lines'].mean(),1),  round(it_test['orig_lines'].mean(),1)],
})
print(summary.to_string(index=False))

## 7. Gold set analysis (ES and IT)

In [ ]:
for df, lang in [(es_gold, 'spanish'), (it_gold, 'italian')]:
    df['original_clean']   = df['original_text'].astype(str).apply(clean_text)
    df['simplified_clean'] = df['simplified_text'].astype(str).apply(clean_text)
    df['orig_tokens']  = df['original_clean'].apply(lambda x: count_tokens(x, lang))
    df['simp_tokens']  = df['simplified_clean'].apply(lambda x: count_tokens(x, lang))
    df['orig_lines']   = df['original_text'].apply(count_lines)
    df['simp_lines']   = df['simplified_text'].apply(count_lines)
    df['ratio_tokens'] = (df['simp_tokens'] / df['orig_tokens']).round(3)

print("Metrics computed for ES and IT gold sets.")

In [ ]:
for name, df in [('ES Gold', es_gold), ('IT Gold', it_gold)]:
    print(f"\n{'='*60}")
    print(f"  {name}  ({len(df)} segments, {df['document_id'].nunique()} documents)")
    print(f"{'='*60}")
    cols = ['orig_tokens','simp_tokens','orig_lines','simp_lines']
    stats = df[cols].describe(percentiles=[0.25,0.5,0.75]).round(2)
    stats.index = ['count','mean','std','min','Q1','median','Q3','max']
    print(stats.to_string())
    print(f"\nMean simplification ratio (simp/orig tokens): {df['ratio_tokens'].mean():.3f}")
    print(f"Ratio > 1 in {(df['ratio_tokens'] > 1).sum()}/{len(df)} segments")

### Gold set: token length distribution (original vs. simplified)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("MER-TRANS 2026 – Gold Set\nToken length distribution: Original vs. Simplified",
             fontsize=13, fontweight='bold')

configs = [
    (es_gold, 'Spanish (ES)', '#4C72B0', '#DD8452', axes[0][0], axes[0][1]),
    (it_gold, 'Italian (IT)', '#4C72B0', '#DD8452', axes[1][0], axes[1][1]),
]

for df, lang_label, col_orig, col_simp, ax_orig, ax_simp in configs:
    max_val = max(df['orig_tokens'].max(), df['simp_tokens'].max()) + 10
    bins = range(0, max_val, 5)
    for ax, col, label, color in [
        (ax_orig, 'orig_tokens', 'Original text',   col_orig),
        (ax_simp, 'simp_tokens', 'Simplified text', col_simp),
    ]:
        ax.hist(df[col], bins=bins, color=color, edgecolor='white', alpha=0.85)
        ax.axvline(df[col].mean(),   color='red',    linestyle='--', linewidth=1.5,
                   label=f"Mean: {df[col].mean():.1f}")
        ax.axvline(df[col].median(), color='orange', linestyle=':',  linewidth=1.5,
                   label=f"Median: {df[col].median():.1f}")
        ax.set_title(f"{lang_label} – {label}")
        ax.set_xlabel('No. tokens')
        ax.set_ylabel('Frequency')
        ax.legend()

plt.tight_layout()
plt.savefig('fig_gold_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### Gold set: mean token length comparison (original vs. simplified)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("MER-TRANS 2026 – Gold Set\nMean token length: Original vs. Simplified",
             fontsize=13, fontweight='bold')

for ax, df, lang_label in [(axes[0], es_gold, 'Spanish (ES)'),
                            (axes[1], it_gold, 'Italian (IT)')]:
    means  = [df['orig_tokens'].mean(), df['simp_tokens'].mean()]
    labels = ['Original', 'Simplified']
    colors = ['#4C72B0', '#DD8452']
    bars = ax.bar(labels, means, color=colors, edgecolor='white', width=0.4)
    ax.bar_label(bars, fmt='%.1f', padding=3)
    ax.set_title(lang_label)
    ax.set_ylabel('Mean tokens')
    ax.set_ylim(0, max(means) * 1.2)

plt.tight_layout()
plt.savefig('fig_gold_mean_comparison.png', dpi=150, bbox_inches='tight')
plt.show()